In [1]:

!pip install datasets pandas scikit-learn


In [2]:

from datasets import load_dataset
import pandas as pd

# Load the Chatbot Arena dataset
dataset = load_dataset(
    "lmarena-ai/arena-human-preference-55k",
    split="train"
)

# Convert to a pandas table
df = dataset.to_pandas()

# Check the dataset
print("Total rows:", len(df))
print("Columns:", df.columns.tolist())

df.head()


README.md:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

train.csv: reconstructing file:   0%|          |  0.00B /  184MB            

train.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/57477 [00:00<?, ? examples/s]

Total rows: 57477
Columns: ['id', 'model_a', 'model_b', 'prompt', 'response_a', 'response_b', 'winner_model_a', 'winner_model_b', 'winner_tie']


,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1,0,0
1,53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0,1,0
2,65089,gpt-3.5-turbo-0613,mistral-medium,"[""explain function calling. how would you call...","[""Function calling is the process of invoking ...","[""Function calling is the process of invoking ...",0,0,1
3,96401,llama-2-13b-chat,mistral-7b-instruct,"[""How can I create a test set for a very rare ...","[""Creating a test set for a very rare category...","[""When building a classifier for a very rare c...",1,0,0
4,198779,koala-13b,gpt-3.5-turbo-0314,"[""What is the best way to travel from Tel-Aviv...","[""The best way to travel from Tel Aviv to Jeru...","[""The best way to travel from Tel-Aviv to Jeru...",0,1,0


In [5]:

# Combine model names from both columns
all_models = pd.concat([
    df["model_a"],
    df["model_b"]
])

# Check our three LLM families
for name in ["gpt", "claude", "gemini"]:
    matches = all_models[
        all_models.str.contains(
            name, case=False, na=False
        )
    ]

    print(name.upper(), ":", len(matches))


GPT : 31840
CLAUDE : 16153
GEMINI : 2924


In [6]:

import pandas as pd
import json

# Identify which LLM family generated the response
def get_llm_family(model):
    model = str(model).lower()

    if "gpt" in model:
        return "GPT"
    elif "claude" in model:
        return "Claude"
    elif "gemini" in model:
        return "Gemini"

    return None

# Convert lists of prompts/responses into text
def convert_to_text(value):
    if isinstance(value, str):
        try:
            value = json.loads(value)
        except (ValueError, TypeError):
            return value

    if isinstance(value, (list, tuple)):
        return "\n".join(str(item) for item in value)

    return ""

# Create our new dataset
rows = []

for _, row in df.iterrows():
    prompt = convert_to_text(row["prompt"])

    for side in ["a", "b"]:
        family = get_llm_family(row[f"model_{side}"])

        if family is not None:
            rows.append({
                "LLM_name": family,
                "LLM_Input": prompt,
                "LLM_output": convert_to_text(
                    row[f"response_{side}"]
                ),
                "source_id": str(row["id"])
            })

clean_df = pd.DataFrame(rows)

print("Dataset created successfully!")
print(clean_df["LLM_name"].value_counts())

clean_df.head()


Dataset created successfully!
LLM_name
GPT       31840
Claude    16153
Gemini     2924
Name: count, dtype: int64


,LLM_name,LLM_Input,LLM_output,source_id
0,GPT,Is it morally right to try to have a certain p...,The question of whether it is morally right to...,30192
1,GPT,Is it morally right to try to have a certain p...,"As an AI, I don't have personal beliefs or opi...",30192
2,GPT,What is the difference between marriage licens...,A marriage license and a marriage certificate ...,53567
3,GPT,explain function calling. how would you call a...,Function calling is the process of invoking or...,65089
4,GPT,What is the best way to travel from Tel-Aviv t...,The best way to travel from Tel-Aviv to Jerusa...,198779


In [7]:

# Remove missing values
clean_df = clean_df.dropna(
    subset=["LLM_name", "LLM_Input", "LLM_output"]
).copy()

# Remove empty prompts and responses
clean_df = clean_df[
    (clean_df["LLM_Input"].str.strip() != "") &
    (clean_df["LLM_output"].str.strip() != "")
]

# Remove duplicate examples
clean_df = clean_df.drop_duplicates(
    subset=["LLM_name", "LLM_Input", "LLM_output"]
)

# Show how many examples remain
print("Total cleaned responses:", len(clean_df))
print("\nResponses by LLM:")
print(clean_df["LLM_name"].value_counts())


Total cleaned responses: 50407

Responses by LLM:
LLM_name
GPT       31499
Claude    16033
Gemini     2875
Name: count, dtype: int64


In [8]:

# Select 2,000 responses from each LLM
balanced_df = (
    clean_df.groupby("LLM_name", group_keys=False)
    .sample(n=2000, random_state=42)
    .reset_index(drop=True)
)

# Check the balanced dataset
print("Total responses:", len(balanced_df))
print("\nResponses per LLM:")
print(balanced_df["LLM_name"].value_counts())

balanced_df.head()


Total responses: 6000

Responses per LLM:
LLM_name
Claude    2000
GPT       2000
Gemini    2000
Name: count, dtype: int64


,LLM_name,LLM_Input,LLM_output,source_id
0,Claude,Please be a great language learning tutor. Spe...,"Привіт! Справи добрі, дякую. Як у вас?\n\nВаше...",2733175726
1,Claude,how to squeeze bananas ?,Here are the steps to squeeze bananas:\n\n1. S...,2779359913
2,Claude,Recommend a workout for weight loss that inclu...,Here is a recommended workout for weight loss ...,25927022
3,Claude,write me all commands to kill all between 7860...,Here are the commands to kill all processes be...,2622543309
4,Claude,Explain relativity. Use the appropriate formula.,Here is a brief explanation of relativity alon...,3398136166


In [10]:

from sklearn.model_selection import GroupShuffleSplit

# Keep identical prompts together
balanced_df["prompt_group"] = (
    balanced_df["LLM_Input"]
    .str.strip()
    .str.lower()
)

# First split: 80% training, 20% remaining
split1 = GroupShuffleSplit(
    n_splits=1, test_size=0.20, random_state=42
)

train_idx, remaining_idx = next(
    split1.split(
        balanced_df,
        groups=balanced_df["prompt_group"]
    )
)

train_df = balanced_df.iloc[train_idx].copy()
remaining_df = balanced_df.iloc[remaining_idx].copy()

# Second split: remaining data into validation and test
split2 = GroupShuffleSplit(
    n_splits=1, test_size=0.50, random_state=42
)

val_idx, test_idx = next(
    split2.split(
        remaining_df,
        groups=remaining_df["prompt_group"]
    )
)

val_df = remaining_df.iloc[val_idx].copy()
test_df = remaining_df.iloc[test_idx].copy()

# Check results
for name, data in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Testing", test_df)
]:
    print(f"\n{name}: {len(data)} responses")
    print(data["LLM_name"].value_counts())

# Check that prompts do not overlap
train_prompts = set(train_df["prompt_group"])
val_prompts = set(val_df["prompt_group"])
test_prompts = set(test_df["prompt_group"])

assert train_prompts.isdisjoint(val_prompts)
assert train_prompts.isdisjoint(test_prompts)
assert val_prompts.isdisjoint(test_prompts)

print("\nSuccess! No prompts overlap between splits.")



Training: 4804 responses
LLM_name
GPT       1616
Gemini    1608
Claude    1580
Name: count, dtype: int64

Validation: 605 responses
LLM_name
Claude    218
GPT       199
Gemini    188
Name: count, dtype: int64

Testing: 591 responses
LLM_name
Gemini    204
Claude    202
GPT       185
Name: count, dtype: int64

Success! No prompts overlap between splits.


In [11]:

# Reduce to 1,000 responses per LLM
small_df = (
    clean_df.groupby("LLM_name", group_keys=False)
    .sample(n=1000, random_state=42)
    .reset_index(drop=True)
)

print("Total responses:", len(small_df))
print(small_df["LLM_name"].value_counts())


Total responses: 3000
LLM_name
Claude    1000
GPT       1000
Gemini    1000
Name: count, dtype: int64


In [12]:

from sklearn.model_selection import GroupShuffleSplit

# Use our NEW smaller dataset
small_df["prompt_group"] = (
    small_df["LLM_Input"].str.strip().str.lower()
)

# Split into training and remaining data
split1 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, remaining_idx = next(
    split1.split(
        small_df,
        groups=small_df["prompt_group"]
    )
)

train_df = small_df.iloc[train_idx].copy()
remaining_df = small_df.iloc[remaining_idx].copy()

# Divide remaining data into validation and testing
split2 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_idx, test_idx = next(
    split2.split(
        remaining_df,
        groups=remaining_df["prompt_group"]
    )
)

val_df = remaining_df.iloc[val_idx].copy()
test_df = remaining_df.iloc[test_idx].copy()

# Display results
for name, data in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Testing", test_df)
]:
    print(f"\n{name}: {len(data)} responses")
    print(data["LLM_name"].value_counts())

# Check for overlapping prompts
assert set(train_df["prompt_group"]).isdisjoint(
    set(val_df["prompt_group"])
)
assert set(train_df["prompt_group"]).isdisjoint(
    set(test_df["prompt_group"])
)
assert set(val_df["prompt_group"]).isdisjoint(
    set(test_df["prompt_group"])
)

print("\nSuccess! No prompts overlap between datasets.")



Training: 2400 responses
LLM_name
GPT       808
Gemini    807
Claude    785
Name: count, dtype: int64

Validation: 301 responses
LLM_name
GPT       104
Claude    101
Gemini     96
Name: count, dtype: int64

Testing: 299 responses
LLM_name
Claude    114
Gemini     97
GPT        88
Name: count, dtype: int64

Success! No prompts overlap between datasets.


In [13]:

# Save the final 3,000-response dataset
small_df.to_csv("llm_dataset.csv", index=False)

# Save the three datasets
train_df.to_csv("train.csv", index=False)
val_df.to_csv("validation.csv", index=False)
test_df.to_csv("test.csv", index=False)

print("All 4 CSV files saved successfully!")


All 4 CSV files saved successfully!
